# 🌐 ChromaDB Online & Client-Server Architecture

Up until now, we used **Chroma Embedded Mode** (`PersistentClient`), where Chroma ran directly inside our local Python process and wrote to a folder on your hard drive.

In real-world enterprise applications, you want a **centralized Vector Database service** that multiple apps, microservices, or team members can access concurrently over the network.

This is where **Chroma Client-Server** and **Chroma Cloud** come in!

### 🎯 Learning Goals:
1. **Understand Embedded vs. Client-Server Architecture**.
2. **Connect over HTTP** using `chromadb.HttpClient(host='localhost', port=8000)`.
3. **Understand Network Vectorization**: How embeddings travel over HTTP REST endpoints.
4. **Ingest & Query across the network**: Push `company_knowledge_base.txt` to the server and query it over HTTP.
5. **Transition to Chroma Cloud (`cloud.trychroma.com`)**: How to point this exact code to Chroma's hosted online service with an API key.

---
## 🏗️ Architectural Overview: Embedded vs. Online / Client-Server

```
1. EMBEDDED MODE (PersistentClient)
┌─────────────────────────────────────────────────────────┐
│  Python Script                                          │
│  ├─ Runs embedding model                                │
│  ├─ Runs HNSW vector index in RAM                       │
│  └─ Writes directly to local disk (./my_local_chroma_db)│
└─────────────────────────────────────────────────────────┘
   * Great for prototyping and local CLI tools.
   * Only ONE process can write at a time (SQLite lock).

2. CLIENT-SERVER / ONLINE MODE (HttpClient)
┌─────────────────────────┐         HTTP / REST          ┌─────────────────────────┐
│  Python App 1 (FastAPI) ├─────────────────────────────►│                         │
└─────────────────────────┘                              │  Chroma HTTP Server     │
┌─────────────────────────┐         JSON & Vectors       │  (Docker / AWS / Cloud) │
│  Python App 2 (RAG bot) ├─────────────────────────────►│                         │
└─────────────────────────┘                              │  ├─ Port 8000 (REST)    │
┌─────────────────────────┐                              │  ├─ High-speed Indexing │
│  Frontend / Node.js App ├─────────────────────────────►│  └─ Central Storage     │
└─────────────────────────┘                              └─────────────────────────┘
   * Multiple apps can query and insert simultaneously.
   * Can scale independently with dedicated GPU/CPU servers.
```

---
## Step 1: Connecting to Chroma via HTTP Client

We have a live Chroma HTTP Server running on `localhost:8000`.
Instead of `PersistentClient(path=...)`, we now use **`HttpClient(host=..., port=...)`**.

Let's test the connection by checking the server's **heartbeat** and **version**:

In [1]:
import chromadb

# Connect to ChromaDB over HTTP (Network Client)
client = chromadb.HttpClient(host="localhost", port=8000)

# Check server health (returns server timestamp in nanoseconds)
heartbeat = client.heartbeat()
version = client.get_version()

print(f" Connected to Chroma HTTP Server!")
print(f" Server Version:   {version}")
print(f"💓 Server Heartbeat: {heartbeat} ns")

 Connected to Chroma HTTP Server!
 Server Version:   1.0.0
💓 Server Heartbeat: 1791534831047760000 ns


---
## Step 2: How Does Vectorization Work Over the Network?

When you run Chroma over HTTP or Cloud, where does the text get converted to vectors?

### The Flow:
1. **Your Client Machine** runs `embedding_function(chunks)` (using your local ONNX model or local Ollama).
2. The text is transformed into 384 numbers locally: `[0.024, -0.198, ...]`. 
3. Your client sends a **JSON payload over HTTP** to `http://localhost:8000/api/v2/tenants/default_tenant/databases/default_database/collections/.../add` containing the vectors, documents, and metadatas.
4. The server receives the vectors, updates its HNSW graph, and sends back an HTTP `200 OK`.

> 💡 **Why this is powerful for security**:
> If you host Chroma on AWS or Chroma Cloud, you can keep your embedding model on your secure machine. The cloud only receives the numerical vectors and never processes your text through third-party APIs!

--- 
## Step 3: Create an Online Collection & Ingest `company_knowledge_base.txt`

Let's create a collection on the remote server and push our 25 company chunks across the network.

In [2]:
import os
import re

# 1. Read the text file
file_path = os.path.join(os.getcwd(), "company_knowledge_base.txt")
with open(file_path, "r", encoding="utf-8") as f:
    raw_content = f.read()

# 2. Parse into chunks with metadata
blocks = raw_content.split("--------------------------------------------------------------------------------")
chunks = []
metadatas = []
ids = []
chunk_num = 1

section_headers = ["Overview:", "Products & Technology:", "Financials & Growth:", "Partnerships & Future Outlook:", "Risk Factors & Challenges:"]

for block in blocks:
    match_name = re.search(r"\[COMPANY:\s*(.*?)\]", block)
    if not match_name:
        continue
    cname = match_name.group(1).strip()
    
    for i, header in enumerate(section_headers):
        pattern = re.escape(header) + r"(.*?)(?=" + "|".join([re.escape(h) for h in section_headers[i+1:]] + ["$"]) + ")"
        sm = re.search(pattern, block, re.DOTALL)
        if sm:
            clean_h = header.replace(":", "").strip()
            chunk_text = f"[{cname} - {clean_h}]\n{sm.group(1).strip()}"
            chunks.append(chunk_text)
            metadatas.append({"company": cname, "section": clean_h})
            ids.append(f"net_chunk_{chunk_num:03d}")
            chunk_num += 1

print(f" Parsed {len(chunks)} chunks from text file.")

# 3. Create collection on the HTTP server
online_collection = client.get_or_create_collection(
    name="online_enterprise_data",
    metadata={"environment": "client-server-demo"}
)

# 4. Transmit across HTTP
online_collection.upsert(ids=ids, documents=chunks, metadatas=metadatas)

print(f" Transmitted and indexed {online_collection.count()} chunks on the server over HTTP!")

 Parsed 25 chunks from text file.
 Transmitted and indexed 25 chunks on the server over HTTP!


--- 
## Step 4: Querying the Online Server Over HTTP

Let's test sending a semantic search request across the network.  
Our query will travel over HTTP, the server will search its HNSW index, and send back the results in JSON format.

In [3]:
query = "Which company develops GPU cluster orchestration and Kubernetes for AI inference?"

# Query travels over HTTP
results = online_collection.query(
    query_texts=[query],
    n_results=2
)

print(f"🔎 Query: '{query}'\n")
for i in range(len(results["ids"][0])):
    print(f"Result #{i+1} (Cosine Distance: {results['distances'][0][i]:.4f}):")
    print(f"Company: {results['metadatas'][0][i]['company']}")
    print(f"Section: {results['metadatas'][0][i]['section']}")
    print(f"Content:\n{results['documents'][0][i]}\n")

🔎 Query: 'Which company develops GPU cluster orchestration and Kubernetes for AI inference?'

Result #1 (Cosine Distance: 0.8109):
Company: CloudScale Systems
Section: Overview
Content:
[CloudScale Systems - Overview]
CloudScale Systems, based in Seattle, Washington, delivers cloud infrastructure software, distributed Kubernetes management, and enterprise-grade GPU cluster orchestration designed specifically for large language model (LLM) fine-tuning and inference.

Result #2 (Cosine Distance: 0.9291):
Company: CloudScale Systems
Section: Products & Technology
Content:
[CloudScale Systems - Products & Technology]
- ScaleKube Enterprise: Production-grade container management platform optimized for zero-downtime multi-cloud deployments.
- NeuroInfer Mesh: A GPU virtualization layer that maximizes GPU utilization by dynamically allocating tensor cores across parallel inference workloads.
- SecureVPC Guard: Automated zero-trust network encryption and compliance monitoring for enterprise da

--- 
## Step 5: Transitioning to Chroma Cloud (`cloud.trychroma.com`)

Chroma Cloud is Chroma's official managed SaaS (exactly like MongoDB Atlas).

### How to Connect to Chroma Cloud:
1. Go to **[cloud.trychroma.com](https://cloud.trychroma.com)** and create an account.
2. Create a Database (e.g. `production-db`).
3. Generate an API Key under Settings.
4. In Python, your code is **100% identical**, you only update the `client` initialization:

```python
import chromadb

# Connect to official Chroma Cloud
cloud_client = chromadb.HttpClient(
    ssl=True,
    host="api.trychroma.com",
    tenant="your-tenant-uuid",      # Given in your Chroma Cloud dashboard
    database="your-database-name",   # e.g., 'production-db'
    headers={"x-chroma-token": "YOUR_CHROMA_API_KEY"}
)

# Now use it exactly like before!
collection = cloud_client.get_or_create_collection("company_data")
collection.upsert(ids=ids, documents=chunks, metadatas=metadatas)
results = collection.query(query_texts=["..."], n_results=3)
```

--- 
## 💡 Summary Comparison

| Feature | Embedded (`PersistentClient`) | Local Server (`HttpClient`) | Chroma Cloud (`cloud.trychroma.com`) |
| :--- | :--- | :--- | :--- |
| **Where it runs** | Inside your Python process | Dedicated local service (port 8000) | Managed Cloud Infrastructure |
| **Storage** | Local hard drive folder | Server's volume (`./server_db`) | Distributed Cloud Storage |
| **Concurrency** | Single-process | Multi-client concurrent access | High-availability global cluster |
| **API Keys** | Not required | Optional auth token | Required (`x-chroma-token`) |
| **Code Changes** | `PersistentClient(path=...)` | `HttpClient(host='localhost', port=8000)` | `HttpClient(host='api.trychroma.com', ...)` |